In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline

model_name = "google-bert/bert-base-multilingual-uncased"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Pipeline approach
# fill_mask_pipeline = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# print("\n--- Fill Mask Example (Pipeline Way) ---")

# text_pipeline_1 = f"Je to veľmi pekný {tokenizer.mask_token}."
# prediction_pipeline_1 = fill_mask_pipeline(text_pipeline_1)
# print(f"\nInput: '{text_pipeline_1}'")
# print(f"Prediction: {prediction_pipeline_1}")

# text_pipeline_2 = f"Dnes je krásny {tokenizer.mask_token}."
# prediction_pipeline_2 = fill_mask_pipeline(text_pipeline_2)
# print(f"\nInput: '{text_pipeline_2}'")
# print(f"Prediction: {prediction_pipeline_2}")

print("\n--- Manual Fill Mask Example ---")

def get_manual_fill_mask(text, model, tokenizer, device, top_k=5):
    # 1. Tokenize the input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding='max_length',
        max_length=512
    ).to(device)

    # 2. Perform model inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits # [batch_size, sequence_length, vocab_size]

    # Find the position of the mask token
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    if mask_token_index.shape[0] == 0:
        return {"error": "No mask token found in the input text."}

    # Extract logits for the masked token (assuming a single mask token for simplicity)
    mask_logits = logits[0, mask_token_index[0], :]

    # Apply softmax to get probabilities
    probabilities = torch.softmax(mask_logits, dim=-1)

    # Get the top_k predicted tokens and their scores
    top_k_scores, top_k_indices = torch.topk(probabilities, top_k)

    predictions = []
    for score, index in zip(top_k_scores, top_k_indices):
        predictions.append(
            {
                "score": score.item(),
                "token": tokenizer.decode(index.item()),
                "token_id": index.item()
            }
        )
    return predictions

# Example 1: Simple sentence with one mask
text_manual_1 = f"Dnes je krásny {tokenizer.mask_token}."
prediction_manual_1 = get_manual_fill_mask(text_manual_1, model, tokenizer, device)
print(f"\nInput (manual way): '{text_manual_1}'")
print(f"Prediction (manual way): {prediction_manual_1}")

# Example 2: Another sentence with a mask
text_manual_2 = f"Chcem si prečítať dobrú {tokenizer.mask_token}."
prediction_manual_2 = get_manual_fill_mask(text_manual_2, model, tokenizer, device)
print(f"\nInput (manual way): '{text_manual_2}'")
print(f"Prediction (manual way): {prediction_manual_2}")

# Example 3: More complex sentence
text_manual_3 = f"Bol to veľmi zaujímavý {tokenizer.mask_token} o slovenskej histórii."
prediction_manual_3 = get_manual_fill_mask(text_manual_3, model, tokenizer, device)
print(f"\nInput (manual way): '{text_manual_3}'")
print(f"Prediction (manual way): {prediction_manual_3}")

Loading google-bert/bert-base-multilingual-uncased...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-multilingual-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Manual Fill Mask Example ---

Input (manual way): 'Dnes je krásny [MASK].'
Prediction (manual way): [{'score': 0.06977751106023788, 'token': 'domov', 'token_id': 92362}, {'score': 0.05332067236304283, 'token': 'hotel', 'token_id': 12752}, {'score': 0.027497276663780212, 'token': 'klub', 'token_id': 14076}, {'score': 0.025568928569555283, 'token': 'dom', 'token_id': 15161}, {'score': 0.01946055144071579, 'token': 'park', 'token_id': 10826}]

Input (manual way): 'Chcem si prečítať dobrú [MASK].'
Prediction (manual way): [{'score': 0.15708687901496887, 'token': 'pomoc', 'token_id': 27151}, {'score': 0.0825238898396492, 'token': 'cestu', 'token_id': 80138}, {'score': 0.06622316688299179, 'token': 'pozornost', 'token_id': 93577}, {'score': 0.05986948683857918, 'token': 'cenu', 'token_id': 47097}, {'score': 0.045100584626197815, 'token': 'moc', 'token_id': 24859}]

Input (manual way): 'Bol to veľmi zaujímavý [MASK] o slovenskej histórii.'
Prediction (manual way): [{'score': 0.1389027088

In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
